"""
GCI study with CODA_SINGLE rings: 3 flight conditions x 5 meshes.

Layout (one GANDALF root per flight condition, as in the hand-made
``GCI/aoa_1.000_m_0.8`` case)::

    GCI/
    ├── aoa_1.000_m_0.8/
    │   ├── outputs/f0.8, f1, f2, f5, f10/   (run_sst_v4.py + run.sh + mesh)
    │   └── metadata/cases_metadata.json     (FRODO-readable, mesh_map incl.)
    ├── aoa_2.500_m_0.9/
    └── aoa_4.000_m_1.05/

The ``mesh`` factor is a design variable, so every row of ``df_cases``
is unique.  Folders only first; node assignment + Slurm submission run
at the end (``submit_cases`` actually calls ``sbatch`` — review before
running this script).
"""

In [8]:
import os
import pandas as pd
from FotR import GANDALF

# --------------------------------------------------------------------------
# Adjustable inputs
# --------------------------------------------------------------------------
DATASET_DIR = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON'
GCI_DIR = os.path.join(DATASET_DIR, 'GCI')
SOURCES_DIR = os.path.join(DATASET_DIR, 'sources')


In [9]:
# Flight conditions (aoa, mach). 
# FLCC = [(1.0, 0.8), (2.5, 0.9), (4.0, 1.05)]
df_post = pd.read_csv(
    '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_#280/metadata/df_post.csv',
    sep=',',
    index_col=0   
)

import plotly.express as px, os
import numpy as np
from typing import Union

def plot_df_post(df_post, col1:str, col2:str, col3:str = None, line_xy:bool = True, save_path:Union['str', bool] = False):
    
    df_post['size'] = df_post['densityresidual_scaled_stage1'].apply(lambda x: 1/-np.log10(x))
    
    df_post['size'] = 10*(df_post['size'] - df_post['size'].min()) / (df_post['size'].max() - df_post['size'].min())
    dicc_batch = {
        'dataset_0': 'batch_0',
        'dataset_1': 'batch_0',
        'dataset_2': 'batch_1',
        'dataset_3': 'batch_2',
    }
    
    df_post['batch'] = df_post['dataset'].apply(lambda x: dicc_batch[x])
    if col3 is not None:
        fig = px.scatter(
            df_post,
            x=col1, y=col2,
            color=col3,
            symbol="dataset",
            symbol_sequence=["circle", "diamond", "triangle-up", "square"],
            hover_data=df_post.columns,
            #tamaño del marker inversamente proporcional al valor de la columna densityresidual_stage0 
            size=df_post['size']
        )
    else:
        fig = px.scatter(
            df_post, 
            x=col1, y=col2, 
            symbol="dataset",
            symbol_sequence=["circle", "diamond", "triangle-up", "square"],
            hover_data=df_post.columns
            )
    
    if line_xy:
        min_val = min(df_post[col1].min(), df_post[col2].min())
        max_val = max(df_post[col1].max(), df_post[col2].max())
        fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
                      line=dict(color='Red', dash='dash'))
    
    fig.update_layout(
        title=f'{col1} vs {col2}',
        xaxis_title=col1,
        yaxis_title=col2,
        height=500, #
        width=800
        )
    # poner la leyenda a la izquierda
    fig.update_layout(legend=dict(x=0, y=1))
    #guardar figura como png si save_path es un string
    if save_path:
        fig.write_image(save_path)
        
    else:
        fig.show()

save_path = None
plot_df_post(df_post, col1='mach', col2='aoa', col3='batch', line_xy=False, save_path=os.path.join(save_path, 'cl0_vs_cl1.png') if save_path is not None else save_path)

intervals = [(0.3, 0.8), (0.8, 1.1), (1.1, 1.4)]
mask = []
p = []
q=0
for inter in intervals:
    min_val, max_val = inter
    mask_inter = (df_post['mach'] >= min_val) & (df_post['mach'] < max_val)
    df_inter = df_post.loc[mask_inter]
    p_error_idx = df_inter['densityresidual_scaled_stage1'].quantile(q)
    p.append(p_error_idx)
    mask = mask + df_inter[df_inter['densityresidual_scaled_stage1'] == p_error_idx].index.tolist()
display(df_post.loc[mask, ['aoa', 'mach', 'densityresidual_scaled_stage1']])

FLCC = [(df_post.loc[idx, 'aoa'], df_post.loc[idx, 'mach']) for idx in mask]

,aoa,mach,densityresidual_scaled_stage1
index,,,
65,3.303404,0.541381,9.940966e-07
100,0.022154,1.071456,4.081985e-06
0,0.022154,1.385825,9.608477e-07


In [10]:

# Mesh refinement factor -> mesh file (order matters: folder f* per factor).
keys = [0.8, 0.9, 1.0, 1.5, 2.0, 3.0, 6.0, 10.0]
MESHES = {key: 'mesh_f22_v8_f' + str(key) + '.msh' for key in keys}
# MESHES = {
#     0.8: 'mesh_TIFON_v8_f0.8.msh',
#     0.9: 'mesh_TIFON_v8_f0.9.msh',
#     1.0: 'mesh_TIFON_v8_f1.0.msh',
#     1.5: 'mesh_TIFON_v8_f1.5.msh',
#     2.0: 'mesh_TIFON_v8_f2.0.msh',
#     3.0: 'mesh_TIFON_v8_f3.0.msh',
#     6.0: 'mesh_TIFON_v8_f6.0.msh',
#     10: 'mesh_TIFON_v8_f10.0.msh',
# }
FILE_SH = 'run.sh'

CHORD = '0.3425'
CM = '-0.085625'
ALT = '11000'

In [12]:
mesh_factors = list(MESHES)
rings = []
for aoa, mach in FLCC:
    root = os.path.join(GCI_DIR, f'aoa_{aoa:.3f}_m_{mach:.3f}')
    gdf = GANDALF(
        root,
        eq_type='rans',
        num_stages=2,
        version='flowsimulator2024',
        ring='coda_single',
    )
    gdf.define_cases(
        method='external',
        external_dataframe=pd.DataFrame({
            'aoa': [aoa] * len(mesh_factors),
            'mach': [mach] * len(mesh_factors),
            'mesh': mesh_factors,
        }),
    )
    gdf.generate_folders(
        base_files=['run_sst_v4.py', 'run.sh'],
        mesh_paths=[
            os.path.join(GCI_DIR, 'meshes', MESHES[m])
            for m in mesh_factors
        ],
        script_dir=SOURCES_DIR,
        folder_fmt='f{mesh:g}',
        overwrite=False,
        update_base_files=True,
        data_to_update={
            'AOA_PLACEHOLDER': 'aoa',
            'MACH_PLACEHOLDER': 'mach',
            'ALT_PLACEHOLDER': ALT,
            'PYTHON_FILE_PLACEHOLDER': 'run_sst_v4.py',
            'CHORD_PLACEHOLDER': CHORD,
            'CM_PLACEHOLDER': CM,
        },
    )
    rings.append(gdf)
    print(f'Prepared {root}: {gdf.folders_name}')

Prepared /home/m.jaraiz/Documentos/DATASETS/data_TIFON/GCI/aoa_3.303_m_0.541: ['f0.8', 'f0.9', 'f1', 'f1.5', 'f2', 'f3', 'f6', 'f10']
Prepared /home/m.jaraiz/Documentos/DATASETS/data_TIFON/GCI/aoa_0.022_m_1.071: ['f0.8', 'f0.9', 'f1', 'f1.5', 'f2', 'f3', 'f6', 'f10']
Prepared /home/m.jaraiz/Documentos/DATASETS/data_TIFON/GCI/aoa_0.022_m_1.386: ['f0.8', 'f0.9', 'f1', 'f1.5', 'f2', 'f3', 'f6', 'f10']


In [13]:
# Slurm submission (cf. example_gandalf_TIFON.ipynb).
NODES = [f'n00{n}' for n in [1, 6, 7]]
CPUS_PER_JOB = 48
for gdf in rings:
    gdf.assign_jobs(
        file_sh=FILE_SH,
        nodes=NODES,
        cpus_per_job=CPUS_PER_JOB,
        submit=True,
    )

In [14]:
for gdf in rings:
    gdf.submit_cases()
